In [7]:
# ============================================================
# EM RV PAIRS – Z-SPREAD RV MODEL (SHORT SAMPLE ADAPTED)
# Engle–Granger (soft) + rolling z-score
# Excel layout: merged ticker cell, then Date | Z-spread
# ============================================================

import os
import numpy as np
import pandas as pd

from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.stattools import adfuller

# =========================
# PATHS & OUTPUTS
# =========================
FILE_PATH = r"C:\Users\thoma\Desktop\Investments\Trading\Projects\Mena_Bond_RV_Version_2/EM_RV_Data_Updated_Apr_03.xlsx"

OUT_DIR = "."
os.makedirs(OUT_DIR, exist_ok=True)

OUT_ALL = os.path.join(OUT_DIR, "rv_em_all.csv")
OUT_NO_BHRAIN = os.path.join(OUT_DIR, "rv_em_no_bahrain.csv")
OUT_KSA = os.path.join(OUT_DIR, "rv_em_ksa_cross_country.csv")

# =========================
# MODEL PARAMETERS (ADAPTED TO SHORT EM SAMPLE)
# =========================
ADF_PVAL_CUTOFF = 0.20
ZSCORE_WINDOW = 30
MIN_OBS = 35

# =========================
# HARD-CODED CLASSIFICATION
# =========================
IG = {"UAE", "QPETRO", "QATAR", "OMAN", "KUWIB", "KSA", "ISRAEL", "ARAMCO", "ADNOCM"}
HY = {"MOROC", "JORDAN", "EGYPT", "BHRAIN"}
QUASI = {"TAQAUH", "PIFKSA", "DPWDU"}

def issuer_bucket(ticker: str) -> str:
    t = ticker.upper()
    for k in IG:
        if k in t:
            return "IG"
    for k in HY:
        if k in t:
            return "HY"
    for k in QUASI:
        if k in t:
            return "QUASI"
    return "OTHER"

def is_bahrain(ticker: str) -> bool:
    return "BHRAIN" in ticker.upper()

def is_ksa(ticker: str) -> bool:
    return "KSA" in ticker.upper()

# =========================
# PARSE EXCEL (MERGED TICKERS)
# =========================
def parse_em_excel(path: str) -> pd.DataFrame:
    raw = pd.read_excel(path, header=None)

    spreads = {}
    col = 0

    while col < raw.shape[1] - 1:
        header = raw.iloc[0, col]

        if pd.isna(header):
            col += 1
            continue

        ticker = str(header).strip()

        df = raw.iloc[1:, [col, col + 1]].copy()
        df.columns = ["date", "spread"]

        df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)
        df["spread"] = pd.to_numeric(df["spread"], errors="coerce")

        s = df.dropna().set_index("date")["spread"].sort_index()

        if len(s) >= MIN_OBS:
            spreads[ticker] = s

        col += 2

    return pd.concat(spreads, axis=1).sort_index()

spread_df = parse_em_excel(FILE_PATH)

print(f"{spread_df.shape[1]} bonds loaded")
print(f"Date range: {spread_df.index.min().date()} → {spread_df.index.max().date()}")

# =========================
# COINTEGRATION / RV LOOP
# =========================
results = []
bonds = list(spread_df.columns)

for i in range(len(bonds)):
    for j in range(i + 1, len(bonds)):

        y_name = bonds[i]
        x_name = bonds[j]

        df = spread_df[[y_name, x_name]].dropna()
        if len(df) < MIN_OBS:
            continue

        y = df[y_name].values
        x = df[x_name].values

        # -------------------------
        # Regression
        # -------------------------
        reg = OLS(y, add_constant(x)).fit()
        alpha, beta = reg.params

        # -------------------------
        # Beta sanity filter
        # -------------------------
        if not (0.3 <= abs(beta) <= 3.0):
            continue

        resid = y - (alpha + beta * x)

        # -------------------------
        # Soft ADF filter
        # -------------------------
        try:
            pval = adfuller(resid)[1]
        except Exception:
            continue

        if pval > ADF_PVAL_CUTOFF:
            continue

        resid_s = pd.Series(resid, index=df.index)

        if len(resid_s) < ZSCORE_WINDOW + 5:
            continue

        mu = resid_s.rolling(ZSCORE_WINDOW).mean()
        sd = resid_s.rolling(ZSCORE_WINDOW).std(ddof=0)
        z = (resid_s - mu) / sd

        z_last = z.iloc[-1]
        if not np.isfinite(z_last):
            continue

        idea = "Long Y / Short X" if z_last > 0 else "Short Y / Long X"

        results.append({
            "y": y_name,
            "x": x_name,
            "idea": idea,

            "beta": round(beta, 2),
            "alpha": round(alpha, 2),
            "z_last": round(z_last, 2),
            "abs_z": round(abs(z_last), 2),
            "adf_pval": round(pval, 4),
            "nobs": len(df),

            "bucket_y": issuer_bucket(y_name),
            "bucket_x": issuer_bucket(x_name),

            "is_bahrain_pair": is_bahrain(y_name) or is_bahrain(x_name),
            "is_ksa_pair": is_ksa(y_name) or is_ksa(x_name),
            "ksa_cross": (is_ksa(y_name) ^ is_ksa(x_name)),
        })

# =========================
# OUTPUTS
# =========================
res = pd.DataFrame(results)

if res.empty:
    print("No RV pairs found – try relaxing parameters further.")
else:
    res = res.sort_values(["abs_z", "adf_pval"], ascending=[False, True]).reset_index(drop=True)

    res.to_csv(OUT_ALL, index=False)
    print(f"Saved: {OUT_ALL}")

    res[~res["is_bahrain_pair"]].to_csv(OUT_NO_BHRAIN, index=False)
    print(f"Saved: {OUT_NO_BHRAIN}")

    res[res["ksa_cross"]].to_csv(OUT_KSA, index=False)
    print(f"Saved: {OUT_KSA}")

    display(res.head(15))


164 bonds loaded
Date range: 2025-10-08 → 2026-04-03
Saved: .\rv_em_all.csv
Saved: .\rv_em_no_bahrain.csv
Saved: .\rv_em_ksa_cross_country.csv


,y,x,idea,beta,alpha,z_last,abs_z,adf_pval,nobs,bucket_y,bucket_x,is_bahrain_pair,is_ksa_pair,ksa_cross
0,BHRAIN 7 1/2 09/20/47,TAQAUH 4 3/4 03/09/37,Short Y / Long X,2.22,65.59,-2.52,2.52,0.1233,122,HY,QUASI,True,False,False
1,PIFKSA 5 3/8 01/29/54,KSA 4 1/2 04/22/60,Long Y / Short X,1.01,5.39,2.49,2.49,0.1895,122,IG,IG,False,True,False
2,BHRAIN 7 1/2 09/20/47,OMAN 6 1/2 03/08/47,Short Y / Long X,2.19,-75.95,-2.48,2.48,0.1949,122,HY,IG,True,False,False
3,BHRAIN 7 1/2 09/20/47,UAE 2 7/8 10/19/41,Short Y / Long X,1.87,133.49,-2.45,2.45,0.0982,122,HY,IG,True,False,False
4,BHRAIN 5.45 09/16/32,BHRAIN 6 5/8 10/06/37,Long Y / Short X,1.02,-43.47,2.43,2.43,0.1194,122,HY,HY,True,False,False
5,ESCWPC 4.45 08/01/35,BHRAIN 5 5/8 07/05/29,Long Y / Short X,0.54,17.58,2.42,2.42,0.0000,71,OTHER,HY,True,False,False
6,BHRAIN 7 1/2 09/20/47,ARAMCO 4 3/8 04/16/49,Short Y / Long X,2.13,-23.43,-2.40,2.40,0.1657,122,HY,IG,True,False,False
7,OMAN 6 1/2 03/08/47,OMAN 6 3/4 01/17/48,Long Y / Short X,0.95,2.98,2.39,2.39,0.0322,122,IG,IG,False,False,False
8,DPWDU 6.85 07/02/37,KSA 4 1/2 04/22/60,Long Y / Short X,1.42,-103.60,2.39,2.39,0.1427,122,QUASI,IG,False,True,True
9,ARAMCO 5 7/8 07/17/64,ADNOCM 5 1/8 09/11/54,Short Y / Long X,0.77,94.93,-2.38,2.38,0.0008,121,IG,IG,False,False,False
